# Notebook05 staged training runner

Purpose: reusable paired ViT/semantic training for three distinct stages: longer learning at128pixels, resolution256pixels, and expanded audited training data at128pixels. Inputs: JSON-normalized immutable stage plan, audited data bundle, fixed probe training IDs and an optional persisted deadline. Outputs: per-arm exact-resume checkpoints, immutable300/1000-update full-state snapshots, paired exposure/trace checks and cumulative timing evidence.

This definition notebook runs in seconds and performs only a pure JSON normalization check. **It does not construct models, execute optimizer steps, launch a worker or run GPU calibration.** The user-facing Notebook05 invokes preparation, actual-config full-state4-versus2+2 verification and training only when the user clicks Run All.

Checkpoint location: `checkpoints/<run_id>/stages/<stage_id>/arms/<arm_id>/`; immutable milestones are under each arm's `milestones/`. Latest two valid resumable checkpoints and best recovery checkpoint remain retained. Save at every epoch boundary, every50 successful steps and at least every15minutes through the inherited training helper. Normal continuation uses the rolling manifest; milestone snapshots retain the same fixed1000-update schedule.

Resume: restart the project kernel and Run All in the registered consumer notebook with identical run ID, stage plan, source hashes and deadline. Exact notebook/cell names are carried in each plan and manifest. Runtime for actual stages is determined from user-executed calibration; no runtime measurement is performed during this build. Official heldout data stay reserved.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot_path = ROOT / "fmm_jepa/bootstrap.ipynb"
_boot_ns = {"__file__": str(_boot_path)}
for _cell in nbformat.read(_boot_path, as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, str(_boot_path), "exec"), _boot_ns)
_boot_ns["install_notebook_package"](ROOT)


## Stable plan identity and paired initialization

In [ ]:
"""Notebook05 staged training definitions. Importing this module never trains."""
from pathlib import Path
from contextlib import nullcontext
import copy, gc, hashlib, json, math, os, shutil, time, uuid
import numpy as np
import torch
from fmm_jepa.semantic_pilot import build_model, initialize_models, run_model, deadline_guard
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer
from fmm_jepa.food101_training import (training_protocol, validate_training_manifest,
    isolated_rng, _hash, _atomic_torch, _rng, _restore_rng, _identity)
from fmm_jepa.food101_evaluation import encoder_state_hash
from fmm_jepa.semantic_evaluation import _release_semantic

RESUME_NOTEBOOK='notebooks/05_Learning_and_Resolution_Scaling.ipynb'
RESUME_CELL='launch-staged-study'

def _persist_numerical_backend_stats(artifact,stage_id,arm_id,event):
    """Observation counters only: cumulative in this process, not checkpoint state."""
    import sys
    backend=sys.modules.get('fmm05_stable_attention')
    value={'status':'RECORDED' if backend is not None else 'UNAVAILABLE',
        'stage_id':stage_id,'arm_id':arm_id,'event':event,'updated_unix':time.time(),
        'worker_pid':os.getpid(),
        'counter_scope':'Cumulative current worker process, including calibration/preflight and any earlier stages; resets on process restart. Not checkpoint state or a performance comparison.'}
    if backend is not None:value['statistics']=backend.get_stats()
    atomic_json(Path(artifact)/'numerical_backend_stats.json',value)
    return value

def _strict_training_protocol(config):
    policy=training_protocol(config)
    if policy['max_consecutive_failures']!=1:
        raise ValueError('Repaired Notebook05 requires stop-on-first-failure; skipped attempts are forbidden')
    policy['failure_policy']='First nonfinite attempt stops before optimizer/scheduler/EMA advancement; no examples are silently skipped. Resume replays from the latest clean full-state checkpoint.'
    return policy

def _assert_clean_training_state(state,effective_batch):
    steps=int(state['successful_updates'])
    if int(state['attempted_updates'])!=steps or state.get('failures') or int(state.get('consecutive_failures',0)):
        raise ValueError('Failed-attempt checkpoint cannot enter a paired repaired run; preserve it and use a fresh run ID')
    if int(state['image_exposures'])!=steps*int(effective_batch):
        raise ValueError('Checkpoint image exposures differ from the paired successful-update stream')
    if len(state['traces'])!=steps or any(not row.get('successful',False) for row in state['traces']):
        raise ValueError('Checkpoint trace history contains skipped or missing attempts')
    if len(state['metrics'])!=steps:
        raise ValueError('Checkpoint metrics do not cover every successful update')

def _require_clean_checkpoint(directory,config):
    path=Path(directory)/'manifest.json'
    if not path.exists():return
    manifest=validate_training_manifest(directory)
    if int(manifest['attempted_updates'])!=int(manifest['successful_updates']):
        raise ValueError('Failed-attempt manifest cannot be resumed in a paired repaired run')
    payload=torch.load(Path(directory)/manifest['latest'],map_location='cpu',weights_only=False)
    _assert_clean_training_state(payload['state'],_strict_training_protocol(config)['effective_batch'])
    if int(payload['sampler']['cursor'])!=int(payload['state']['image_exposures']):
        raise ValueError('Checkpoint sampler position differs from consumed image exposures')

def normalize_stage_plan(plan):
    """One JSON-roundtrip representation before hashing, storing or comparing.

    Integer metadata dictionary keys become strings now, before registration,
    so a fresh JSON load cannot change sorted canonical hashing later.
    """
    return json.loads(json.dumps(plan,allow_nan=False))

def stage_definition_hashes(root):
    root=Path(root);result=definition_hashes(root)
    for folder in ('experiments/04_followup','experiments/05_scaling'):
        for path in sorted((root/folder).glob('*.ipynb')):
            nb=json.loads(path.read_text())
            sources=[''.join(c['source']) for c in nb['cells']
                     if c['cell_type']=='code' and 'ahr-export' in c.get('metadata',{}).get('tags',[])]
            result[str(path.relative_to(root))]=hashlib.sha256('\n'.join(sources).encode()).hexdigest()
    return result

def _record(path):
    path=Path(path)
    return {'path':str(path.resolve()),'sha256':digest_file(path)}

def _arm_paths(root,artifact,plan,arm):
    return (Path(artifact)/'arms'/arm['id'],
            Path(root)/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'arms'/arm['id'])

def _status(artifact,stage,state,**details):
    record={'stage':stage,'status':state,'updated_unix':time.time(),**details}
    atomic_json(Path(artifact)/'stage_status.json',record)
    print(stage,state,details.get('arm_id',''),flush=True)
    return record

def _deadline(deadline_unix,reserve=0.):
    if deadline_unix is not None and time.time()+float(reserve)>=float(deadline_unix):
        raise TimeoutError('Persisted Notebook05 deadline reached; valid checkpoints retained')

def _guard(deadline_unix):
    return deadline_guard(deadline_unix) if deadline_unix is not None else nullcontext()

def _plan_binding(root,plan,bundle):
    safe=lambda s:isinstance(s,str) and bool(s) and all(c.isalnum() or c in '-_' for c in s)
    if not safe(plan.get('run_id')) or not safe(plan.get('stage_id')):raise ValueError('Safe run_id and stage_id required')
    arms=plan.get('arms',[])
    if not arms or len({a['id'] for a in arms})!=len(arms) or any(not safe(a['id']) for a in arms):
        raise ValueError('Unique safe arm IDs required')
    updates=int(plan['updates']);milestones=list(map(int,plan.get('milestones',[300,updates])))
    if updates<4 or len(set(milestones))!=len(milestones) or any(m<1 or m>updates for m in milestones):
        raise ValueError('Invalid schedule or milestone set')
    if updates not in milestones:raise ValueError('Final update must be an explicit milestone')
    if int(plan.get('round_updates',100))<1:raise ValueError('Positive round size required')
    train=list(bundle['train_ids']);probe=list(bundle['probe_train_ids']);dev=list(bundle['dev_ids']);heldout=list(bundle.get('heldout_ids',[]))
    if not train or not probe or not dev or any(len(ids)!=len(set(ids)) for ids in (train,probe,dev)):
        raise ValueError('Nonempty unique training/probe/development memberships required')
    if set(train)&set(dev) or set(probe)&set(dev) or (set(train)|set(probe)|set(dev))&set(heldout):
        raise ValueError('Development or official-test leakage')
    if not set(probe).issubset(train):raise ValueError('Fixed probe training set must be nested inside stage pretraining pool')
    if 'train_count' in plan and int(plan['train_count'])!=len(train):raise ValueError('Stage training count differs from plan')
    if 'probe_train_count' in plan and int(plan['probe_train_count'])!=len(probe):raise ValueError('Probe count differs from plan')
    seeds=sorted({int(a['seed']) for a in arms})
    if 'seeds' in plan and sorted(map(int,plan['seeds']))!=seeds:raise ValueError('Declared seed list differs from arms')
    for arm in arms:
        config=arm['config'];policy=_strict_training_protocol(config)
        if arm['kind'] not in ('vit','semantic') or int(config['seeds']['model_init'])!=int(arm['seed']):
            raise ValueError('Architecture or model seed mismatch')
        if policy['updates']!=updates or policy['effective_batch']!=int(plan.get('effective_batch',128)):
            raise ValueError('Schedule/effective batch differs from registered stage')
        if config.get('dtype','float32')!='float32':raise ValueError('Only verified FP32 enabled')
        if config.get('evaluation',{}).get('heldout_enabled',False):raise ValueError('Official test is reserved')
        if int(config['probe']['epochs'])!=20:raise ValueError('Fixed epoch20 probes required')
        if policy['checkpoint_every']>50:raise ValueError('Checkpoint at most every50 successful updates')
    for seed in seeds:
        group=[a for a in arms if int(a['seed'])==seed]
        if sorted(a['kind'] for a in group)!=['semantic','vit']:raise ValueError('Each seed needs exactly one ViT and one semantic arm')
        # Match every training/evaluation choice; architecture is the sole paired-arm distinction.
        configs=[normalize_stage_plan(a['config']) for a in group]
        for config in configs:
            config.get('model',{}).pop('kind',None)
        if configs[0]!=configs[1]:raise ValueError('Paired architectures differ in configuration beyond model kind')
    sources=plan.get('source_hashes',stage_definition_hashes(root));actual=stage_definition_hashes(root)
    if sources!=actual:raise ValueError('Registered Notebook05 definition hashes changed')
    consumer_hash=plan.get('consumer_notebook_sha256')
    if consumer_hash:
        nb=json.loads((Path(root)/plan.get('resume_notebook',RESUME_NOTEBOOK)).read_text())
        code=[''.join(c['source']) for c in nb['cells'] if c['cell_type']=='code']
        if _hash(code)!=consumer_hash:raise ValueError('Registered main notebook code changed')
    return {'schema':'fmm-jepa-notebook05-stage-binding/v1','plan_hash':_hash(plan),
        'source_hashes':sources,'dataset_fingerprint':bundle['dataset'].fingerprint,
        'split_sha256':bundle['split_manifest']['split_sha256'],
        'training_ids_sha256':_hash(train),'training_examples':len(train),
        'probe_training_ids_sha256':_hash(probe),'probe_training_examples':len(probe),
        'development_ids_sha256':_hash(dev),'development_examples':len(dev),'heldout_status':'NOT_RUN'}

def _resume_metadata(directory,plan):
    path=Path(directory)/'manifest.json'
    if path.exists():
        record=json.loads(path.read_text())
        record['resume_notebook']=plan.get('resume_notebook',RESUME_NOTEBOOK)
        record['resume_cell']=plan.get('resume_cell',RESUME_CELL)
        record['resume_instructions']='Fresh kernel, identical run/stage plan and source hashes, Run All; persisted deadlines and successful-update schedule never reset.'
        record['milestone_manifest']='milestones/manifest.json'
        atomic_json(path,record)

def prepare_stage(root,artifact,plan,bundle):
    """CPU preparation at user execution: immutable paired initial states and protocols."""
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    artifact.mkdir(parents=True,exist_ok=True);binding=_plan_binding(root,plan,bundle)
    path=artifact/'stage_registration.json'
    if path.exists():
        registration=json.loads(path.read_text())
        if registration['binding']!=binding or registration['normalized_plan']!=plan:
            raise ValueError('Existing stage registration differs; no silent restart')
    else:registration={'binding':binding,'normalized_plan':plan,'arms':{},'heldout_status':'NOT_RUN'}
    parameter_hashes={}
    for arm in plan['arms']:
        folder,checkpoint=_arm_paths(root,artifact,plan,arm);folder.mkdir(parents=True,exist_ok=True)
        models,initial=initialize_models(arm['config'],folder);model=models[arm['kind']]
        trainable={name:value.detach().cpu() for name,value in model.named_parameters() if value.requires_grad}
        parameter_hash=encoder_state_hash(trainable);seed=int(arm['seed'])
        if seed in parameter_hashes and parameter_hashes[seed]!=parameter_hash:
            raise AssertionError('Paired ViT/semantic trainable initial parameters differ')
        parameter_hashes[seed]=parameter_hash
        protocol={'schema':'fmm-jepa-notebook05-arm/v1','binding':binding,'source_hashes':binding['source_hashes'],
            'arm':arm,'initialization':initial,'training':_strict_training_protocol(arm['config']),
            'milestones':plan['milestones'],'selection':'Predeclared milestone/final full-state checkpoints; fixed probe epoch20',
            'heldout_status':'NOT_RUN'}
        protocol_path=folder/'protocol.json'
        if protocol_path.exists() and json.loads(protocol_path.read_text())!=protocol:raise ValueError('Arm protocol changed')
        atomic_json(protocol_path,protocol)
        registration['arms'][arm['id']]={'protocol_sha256':digest_file(protocol_path),'initialization':initial,
            'initial_teacher':initial['states'][arm['kind']]['teacher'],'trainable_parameter_hash':parameter_hash,
            'checkpoint_directory':str(checkpoint.resolve())}
        del trainable,model,models;gc.collect()
    atomic_json(path,registration)
    return registration

def _registered(root,artifact,plan,bundle):
    plan=normalize_stage_plan(plan);binding=_plan_binding(root,plan,bundle)
    registration=json.loads((Path(artifact)/'stage_registration.json').read_text())
    if registration['binding']!=binding or registration['normalized_plan']!=plan:
        raise ValueError('Stage source/data/plan identity differs; exact resume refused')
    for arm in plan['arms']:
        folder,_=_arm_paths(root,artifact,plan,arm)
        if digest_file(folder/'protocol.json')!=registration['arms'][arm['id']]['protocol_sha256']:
            raise ValueError('Arm protocol checksum differs')
    return registration

def _initial_model(root,artifact,plan,arm):
    folder,_=_arm_paths(root,artifact,plan,arm)
    initial=json.loads((folder/'initialization/manifest.json').read_text())
    record=initial['states'][arm['kind']]['model']
    if digest_file(record['path'])!=record['sha256']:raise ValueError('Initial model checksum differs')
    with isolated_rng(int(arm['seed'])):model=build_model(arm['config'],arm['kind']).cpu()
    model.load_state_dict(torch.load(record['path'],map_location='cpu',weights_only=True))
    return model


## Immutable milestones and exact resume handoff

In [ ]:
def validate_stage_milestones(directory,plan,required=()):
    """Verify every registered immutable milestone before any reuse."""
    directory=Path(directory);path=directory/'milestones/manifest.json'
    if not path.exists():
        if required:raise ValueError('Required milestone manifest is absent')
        return {'schema':'notebook05-milestones/v1','plan_hash':_hash(normalize_stage_plan(plan)),'records':{}}
    manifest=json.loads(path.read_text())
    if manifest.get('schema')!='notebook05-milestones/v1' or manifest['plan_hash']!=_hash(normalize_stage_plan(plan)):
        raise ValueError('Milestone identity changed')
    for step,record in manifest['records'].items():
        expected=directory/'milestones'/f'step_{int(step):06d}.pt'
        if Path(record['path']).resolve()!=expected.resolve() or digest_file(expected)!=record['sha256']:
            raise ValueError('Immutable milestone path/checksum differs')
        if int(step)!=record['successful_updates']:raise ValueError('Milestone step metadata mismatch')
    if any(str(int(step)) not in manifest['records'] for step in required):raise ValueError('Required milestone missing')
    return manifest

def _save_milestone(directory,plan,manifest):
    """Atomic full-state copy outside rolling latest-two/best retention."""
    directory=Path(directory);step=int(manifest['successful_updates'])
    if step not in list(map(int,plan['milestones'])):return
    saved=validate_stage_milestones(directory,plan)
    source=directory/manifest['latest'];source_hash=digest_file(source)
    if str(step) in saved['records']:
        if saved['records'][str(step)]['sha256']!=source_hash:raise ValueError('Attempt to replace an immutable milestone')
        return
    folder=directory/'milestones';folder.mkdir(parents=True,exist_ok=True);destination=folder/f'step_{step:06d}.pt'
    recovered_orphan=destination.exists()
    if recovered_orphan:
        if digest_file(destination)!=source_hash:raise ValueError('Orphan milestone differs from authoritative checkpoint')
    else:
        temporary=folder/(destination.name+'.'+uuid.uuid4().hex+'.tmp')
        try:
            with source.open('rb') as reader,temporary.open('wb') as writer:
                shutil.copyfileobj(reader,writer);writer.flush();os.fsync(writer.fileno())
            if digest_file(temporary)!=source_hash:raise IOError('Milestone copy checksum differs')
            os.replace(temporary,destination)
            descriptor=os.open(folder,os.O_DIRECTORY)
            try:os.fsync(descriptor)
            finally:os.close(descriptor)
        finally:temporary.unlink(missing_ok=True)
    saved['records'][str(step)]={**_record(destination),'successful_updates':step,
        'attempted_updates':int(manifest['attempted_updates']),'image_exposures':int(manifest['image_exposures']),
        'source_checkpoint':str(source.resolve()),'identity':manifest['identity'],
        'full_state_snapshot':True,'immutable':True,'recovered_verified_orphan':recovered_orphan,
        'timing_status':'PENDING_INVOCATION_COMPLETION'}
    saved.update(resume_notebook=plan.get('resume_notebook',RESUME_NOTEBOOK),
                 resume_cell=plan.get('resume_cell',RESUME_CELL),
                 resume_policy='Normal continuation uses the arm rolling manifest; milestones are immutable evaluation/recovery evidence, not schedule extensions.')
    atomic_json(folder/'manifest.json',saved)

def _attach_milestone_performance(directory,plan,result):
    step=int(result['resume']['final_step'])
    if step not in list(map(int,plan['milestones'])):return
    saved=validate_stage_milestones(directory,plan,required=[step]);record=saved['records'][str(step)]
    rows=[row for row in result['metrics'] if row['successful_updates']<=step]
    if len(rows)!=step:raise ValueError('Milestone timing rows do not cover every successful update')
    timing={}
    for field in ('step_seconds','compute_seconds','data_seconds'):
        if any(field not in row for row in rows):raise ValueError('Milestone checkpoint has unflushed timing')
        timing['cumulative_'+field]=float(sum(row[field] for row in rows))
    performance=copy.deepcopy(result['performance'])
    evidence={**timing,'performance':performance,
        'cumulative_training_invocation_wall_seconds':performance.get('wall_seconds_including_checkpoints_and_diagnostics'),
        'timing_status':'COMPLETE' if performance.get('successful_update_coverage_complete') else 'STEP_TIMING_COMPLETE_WALL_HISTORY_UNKNOWN',
        'timing_scope':'Cumulative successful-step data/compute from checkpoint metrics; invocation wall includes checkpoint work and is UNKNOWN if interrupted history cannot be reconstructed.'}
    if record.get('timing_status') in ('COMPLETE','STEP_TIMING_COMPLETE_WALL_HISTORY_UNKNOWN'):
        for field,value in timing.items():
            if record[field]!=value:raise ValueError('Previously published milestone timing changed')
        # Keep first successful recorded history; a load-only invocation is not new training evidence.
        return
    record.update(evidence);atomic_json(Path(directory)/'milestones/manifest.json',saved)

def _run_stage_model(plan,*args,**kwargs):
    """Correct handoff metadata and pin milestones on every upstream checkpoint save."""
    import fmm_jepa.semantic_pilot as pilot
    original=pilot._save_state
    config=args[1] if len(args)>1 else kwargs['config']
    directory=args[4] if len(args)>4 else kwargs['checkpoint_dir']
    _strict_training_protocol(config)
    _require_clean_checkpoint(directory,config)
    def save(*save_args,**save_kwargs):
        state=save_args[1] if len(save_args)>1 else save_kwargs['state']
        _assert_clean_training_state(state,_strict_training_protocol(config)['effective_batch'])
        original(*save_args,**save_kwargs)
        directory=Path(save_args[0]);_resume_metadata(directory,plan)
        manifest=json.loads((directory/'manifest.json').read_text())
        _save_milestone(directory,plan,manifest)
        return manifest
    pilot._save_state=save
    try:return run_model(*args,**kwargs)
    finally:pilot._save_state=original


## Actual-configuration preflight, invoked only at user execution

In [ ]:
def verify_stage_resume(root,artifact,plan,bundle,deadline_unix=None):
    """At user execution only: actual-config four updates equals reconstructed2+2.

    One first-seed certificate per architecture covers that stage resolution,
    batch/accumulation, data membership and full successful-update schedule.
    Every RNG/optimizer/model/scheduler/EMA/sampler/trace is compared bitwise;
    elapsed timing is the only excluded state.
    """
    from fmm_jepa.food101_benchmark import _food_exact_tree
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    registration=_registered(root,artifact,plan,bundle);path=artifact/'stage_preflight.json'
    if path.exists():
        certificate=json.loads(path.read_text())
        if certificate['status']!='PASS' or certificate['binding']!=registration['binding']:
            raise ValueError('Actual-config preflight identity differs')
        for entry in certificate['arms'].values():
            for record in entry['checkpoints'].values():
                folder=Path(record['directory']);manifest=validate_training_manifest(folder)
                if manifest['successful_updates']!=4 or digest_file(folder/manifest['latest'])!=record['sha256']:
                    raise ValueError('Full-state preflight evidence changed')
        return certificate
    selected=[]
    for kind in ('vit','semantic'):
        selected.append(next(a for a in plan['arms'] if a['kind']==kind))
    fields=['model','optimizer','scheduler','ema_position','scaler','precision','rng','sampler',
            'streams','microstep','partial_accumulation','world_size','rank','replay','config','protocol','identity']
    state_fields=['successful_updates','attempted_updates','image_exposures','consecutive_failures','traces','failures','diagnostics']
    timing={'data_seconds','compute_seconds','step_seconds','images_per_second','_cuda_event_count'}
    certificate={'status':'RUNNING','binding':registration['binding'],'arms':{},
        'scope':'First seed per architecture; exact actual stage config/data and full schedule; elapsed timing excluded'}
    original_rng=_rng()
    try:
        with _guard(deadline_unix):
            for arm in selected:
                _deadline(deadline_unix,5);_status(artifact,'exact_resume','RUNNING',arm_id=arm['id'])
                arm_folder,_=_arm_paths(root,artifact,plan,arm)
                protocol=json.loads((arm_folder/'protocol.json').read_text())
                folder=artifact/'preflight'/arm['id'];folder.mkdir(parents=True,exist_ok=True)
                rng_path=folder/'trajectory_rng.pt';rng_record=folder/'trajectory_rng.json'
                if rng_record.exists():
                    if digest_file(rng_path)!=json.loads(rng_record.read_text())['sha256']:raise ValueError('Preflight RNG changed')
                    trajectory=torch.load(rng_path,map_location='cpu',weights_only=False)
                else:
                    if rng_path.exists():raise ValueError('Orphan trajectory RNG; inspect before recovery')
                    with isolated_rng(int(arm['seed'])):trajectory=_rng()
                    _atomic_torch(rng_path,trajectory);atomic_json(rng_record,_record(rng_path))
                reference_dir=root/'checkpoints'/plan['run_id']/'stages'/plan['stage_id']/'preflight'/(arm['id']+'-reference')
                split_dir=reference_dir.with_name(arm['id']+'-split')
                _restore_rng(copy.deepcopy(trajectory))
                reference=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'reference',reference_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                del reference
                _restore_rng(copy.deepcopy(trajectory))
                partial=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=2,deadline_unix=deadline_unix)
                if partial['resume']['final_step'] not in (2,4):raise AssertionError('Preflight cutoff before split boundary')
                del partial
                resumed=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder/'split',split_dir,protocol,
                    model=_initial_model(root,artifact,plan,arm),stop_after=4,deadline_unix=deadline_unix)
                lm,rm=validate_training_manifest(reference_dir),validate_training_manifest(split_dir)
                left=torch.load(reference_dir/lm['latest'],map_location='cpu',weights_only=False)
                right=torch.load(split_dir/rm['latest'],map_location='cpu',weights_only=False)
                for field in fields:_food_exact_tree(left[field],right[field],field)
                for field in state_fields:_food_exact_tree(left['state'][field],right['state'][field],'state.'+field)
                lmetrics=[{k:v for k,v in row.items() if k not in timing} for row in left['state']['metrics']]
                rmetrics=[{k:v for k,v in row.items() if k not in timing} for row in right['state']['metrics']]
                _food_exact_tree(lmetrics,rmetrics,'deterministic_metrics')
                if lm['successful_updates']!=4 or rm['attempted_updates']!=4 or len(rm['checkpoints'])<2:
                    raise AssertionError('Four finite updates and latest-two retention required')
                certificate['arms'][arm['id']]={'status':'PASS','bitwise_equal':True,
                    'image_size':arm['config']['image_size'],'training_examples':len(bundle['train_ids']),
                    'microbatch':arm['config']['training']['microbatch'],'accumulation_steps':arm['config']['training']['accumulation_steps'],
                    'schedule_updates':plan['updates'],'full_state_fields':fields,
                    'checkpoints':{'reference':{'directory':str(reference_dir),'sha256':digest_file(reference_dir/lm['latest'])},
                                   'resumed':{'directory':str(split_dir),'sha256':digest_file(split_dir/rm['latest'])}}}
                atomic_json(folder/'certificate.json',certificate['arms'][arm['id']])
                del left,right,resumed;_release_semantic(arm['config']['device'])
    finally:_restore_rng(original_rng)
    certificate['status']='PASS';atomic_json(path,certificate)
    return certificate


## Fixed-schedule paired stage training

In [ ]:
def train_stage(root,artifact,plan,bundle,deadline_unix=None):
    """At user execution: equal-budget paired training; no post-hoc schedule extension."""
    if not os.environ.get('TMUX'):raise RuntimeError('Long stage training requires detached tmux')
    root,artifact=Path(root),Path(artifact);plan=normalize_stage_plan(plan)
    registration=_registered(root,artifact,plan,bundle)
    for arm in plan['arms']:
        _,directory=_arm_paths(root,artifact,plan,arm)
        _require_clean_checkpoint(directory,arm['config'])
    certificate=json.loads((artifact/'stage_preflight.json').read_text())
    if certificate['status']!='PASS' or certificate['binding']!=registration['binding']:
        raise ValueError('Exact actual-stage full-state preflight must pass before training')
    expected_identities={}
    for arm in plan['arms']:
        folder,_=_arm_paths(root,artifact,plan,arm)
        expected_identities[arm['id']]=_identity(arm['config'],bundle,json.loads((folder/'protocol.json').read_text()))
    training_deadline=None if deadline_unix is None else float(deadline_unix)-float(plan.get('evaluation_reserve_seconds',0))
    budget=int(plan['updates']);chunk=int(plan.get('round_updates',100));milestones=sorted(map(int,plan['milestones']))
    endpoints=sorted(set([min(step,budget) for step in range(chunk,budget+chunk,chunk)]+milestones))
    with run_writer(artifact/'stage_training_writer'),_guard(deadline_unix):
        # Finish a provably identical milestone copy if interruption happened
        # between its atomic training checkpoint and immutable snapshot manifest.
        for arm in plan['arms']:
            _,directory=_arm_paths(root,artifact,plan,arm)
            if (directory/'manifest.json').exists():
                manifest=validate_training_manifest(directory,expected_identities[arm['id']]);_save_milestone(directory,plan,manifest)
                passed=[m for m in milestones if m<=manifest['successful_updates']]
                validate_stage_milestones(directory,plan,required=passed)
        for endpoint in endpoints:
            counts={}
            for arm in plan['arms']:
                _,directory=_arm_paths(root,artifact,plan,arm)
                counts[arm['id']]=(validate_training_manifest(directory,expected_identities[arm['id']])['successful_updates'] if (directory/'manifest.json').exists() else 0)
            if min(counts.values())>=endpoint:continue
            estimate=float(plan.get('pessimistic_seconds_per_round_update',0))*(endpoint-min(counts.values()))
            _deadline(training_deadline,estimate)
            for arm in plan['arms']:
                if counts[arm['id']]>=endpoint:continue
                _deadline(training_deadline)
                folder,directory=_arm_paths(root,artifact,plan,arm)
                protocol=json.loads((folder/'protocol.json').read_text())
                _status(artifact,'training','RUNNING',arm_id=arm['id'],round_endpoint=endpoint,
                        planned_updates=budget,completed_before_round=counts)
                try:
                    result=_run_stage_model(plan,arm['kind'],arm['config'],bundle,folder,directory,protocol,
                        model=_initial_model(root,artifact,plan,arm),stop_after=endpoint,deadline_unix=training_deadline)
                finally:
                    _persist_numerical_backend_stats(artifact,plan['stage_id'],arm['id'],'training_invocation_exit')
                _attach_milestone_performance(directory,plan,result)
                final=int(result['resume']['final_step']);del result;_release_semantic(arm['config']['device'])
                if final!=endpoint:raise TimeoutError('Stage stopped before a complete registered round; no paired quality conclusion')
    records={};seed_traces={}
    for arm in plan['arms']:
        folder,directory=_arm_paths(root,artifact,plan,arm);manifest=validate_training_manifest(directory,expected_identities[arm['id']])
        payload=torch.load(directory/manifest['latest'],map_location='cpu',weights_only=False);state=payload['state']
        if payload['identity']!=expected_identities[arm['id']]:raise ValueError('Final checkpoint belongs to another arm/protocol/environment')
        if payload['scheduler']['position']!=state['successful_updates'] or payload['ema_position']!=state['successful_updates']:
            raise ValueError('Final scheduler/EMA position differs from completed updates')
        if payload['sampler']['cursor']!=state['image_exposures']:raise ValueError('Final sampler position differs from image exposures')
        fields=('sample_ids','occurrences','augmentation_ids','mask_ids')
        trace_hash=_hash([{key:row[key] for key in fields} for row in state['traces']])
        seed_traces.setdefault(int(arm['seed']),[]).append(trace_hash)
        snapshots=validate_stage_milestones(directory,plan,required=milestones)['records']
        # A crash after checkpointing but before wall-history publication is
        # represented explicitly rather than silently synthesizing lost timing.
        for step,record in snapshots.items():
            if record['identity']!=expected_identities[arm['id']]:raise ValueError('Milestone belongs to another arm/protocol/environment')
            if record.get('timing_status')=='PENDING_INVOCATION_COMPLETION':
                checkpoint=torch.load(record['path'],map_location='cpu',weights_only=False)
                metrics=checkpoint['state']['metrics']
                record.update({f'cumulative_{field}':float(sum(row[field] for row in metrics))
                    for field in ('step_seconds','compute_seconds','data_seconds')})
                record.update(performance=None,cumulative_training_invocation_wall_seconds=None,
                    timing_status='STEP_TIMING_COMPLETE_WALL_HISTORY_UNKNOWN')
                del checkpoint
        records[arm['id']]={'arm_id':arm['id'],'kind':arm['kind'],'seed':int(arm['seed']),
            'successful_updates':state['successful_updates'],'attempted_updates':state['attempted_updates'],
            'image_exposures':state['image_exposures'],'failures':state['failures'],'paired_trace_sha256':trace_hash,
            'initial_teacher':registration['arms'][arm['id']]['initial_teacher'],
            'milestones':snapshots,'latest_checkpoint':_record(directory/manifest['latest']),
            'performance':json.loads((folder/(arm['kind']+'_performance.json')).read_text())}
        del payload,state
    steps=[r['successful_updates'] for r in records.values()]
    equal=len(records)==len(plan['arms']) and len(set(steps))==1
    paired=equal and all(len(set(hashes))==1 for hashes in seed_traces.values())
    no_failures=all(not r['failures'] and r['successful_updates']==r['attempted_updates'] for r in records.values())
    complete=paired and no_failures and steps[0]==budget
    summary={'schema':'fmm-jepa-notebook05-training/v1','status':'COMPLETE' if complete else 'INCOMPLETE',
        'stage_id':plan['stage_id'],'binding':registration['binding'],'planned_updates':budget,'milestones':milestones,
        'equal_updates':equal,'paired_traces_within_seed':paired,'no_failed_attempts':no_failures,
        'training_examples':len(bundle['train_ids']),'probe_training_examples':len(bundle['probe_train_ids']),
        'arms':records,'heldout_status':'NOT_RUN',
        'selection':'Every arm finishes the same preregistered schedule; immutable milestones never extend or restart it.'}
    atomic_json(artifact/'stage_training_summary.json',summary)
    if not complete:raise RuntimeError('Stage incomplete/unpaired or failed attempts present; no paired quality conclusion allowed')
    _status(artifact,'training','COMPLETED',planned_updates=budget,arms=len(records))
    return summary


## Pure CPU metadata validation

In [ ]:
# Pure data-structure checks only. No model, optimizer, images, GPU or experiment.
_example={'metadata':{'weighted_losses':{16:0.5,128:0.4}},'updates':1000,'milestones':[300,1000]}
_normalized=normalize_stage_plan(_example)
assert _normalized==json.loads(json.dumps(_example))
assert _hash(_normalized)==_hash(normalize_stage_plan(json.loads(json.dumps(_normalized))))
assert _example['metadata']['weighted_losses']=={16:0.5,128:0.4}
assert _normalized['metadata']['weighted_losses']=={'16':0.5,'128':0.4}
print({'status':'PASS','json_roundtrip_hash_stable':True,'input_unmodified':True,
       'training_executed':False,'optimizer_steps':0,'gpu_work':False})


## Pure CPU checkpoint adapter checks

These temporary byte fixtures exercise atomic milestone copying, immutable snapshot rejection and save-hook restoration. They contain no model/optimizer state, are deleted after validation and are not experiment evidence.

In [ ]:
# Pure CPU mock I/O: validates milestone immutability without a model checkpoint.
import tempfile
_mock_root=ROOT/'artifacts/notebook_05/build_validation';_mock_root.mkdir(parents=True,exist_ok=True)
_mock_plan={'run_id':'MOCK_ONLY','stage_id':'learning','updates':1000,'milestones':[300,1000]}
with tempfile.TemporaryDirectory(prefix='milestone-mock-',dir=_mock_root) as _temporary:
    _directory=Path(_temporary);_source=_directory/'step_000300_attempt_000300.pt'
    _source.write_bytes(b'MOCK ONLY: no model, optimizer, data or experiment state')
    _manifest={'latest':_source.name,'successful_updates':300,'attempted_updates':300,
               'image_exposures':38400,'identity':{'fixture':'PURE_CPU_MOCK'}}
    _save_milestone(_directory,_mock_plan,_manifest)
    _first=validate_stage_milestones(_directory,_mock_plan,required=[300])
    _save_milestone(_directory,_mock_plan,_manifest)
    assert validate_stage_milestones(_directory,_mock_plan,required=[300])==_first
    _pinned=Path(_first['records']['300']['path'])
    assert _pinned.read_bytes()==_source.read_bytes()
    _source.write_bytes(b'CHANGED MOCK SOURCE')
    try:_save_milestone(_directory,_mock_plan,_manifest)
    except ValueError as _error:assert 'immutable milestone' in str(_error)
    else:raise AssertionError('Changed milestone source was accepted')
    assert _pinned.read_bytes()==b'MOCK ONLY: no model, optimizer, data or experiment state'
# The upstream save hook must restore even if the invoked operation raises.
import fmm_jepa.semantic_pilot as _pilot
_real_runner=run_model;_original_save=_pilot._save_state
try:
    def _failing_mock(*args,**kwargs):raise RuntimeError('PURE_CPU_MOCK_SENTINEL')
    run_model=_failing_mock
    try:_run_stage_model(_mock_plan)
    except RuntimeError as _error:assert str(_error)=='PURE_CPU_MOCK_SENTINEL'
    else:raise AssertionError('Mock did not raise')
    assert _pilot._save_state is _original_save
finally:run_model=_real_runner
print({'status':'PASS','mock_milestone_idempotent':True,'changed_snapshot_refused':True,
       'upstream_save_hook_restored_on_exception':True,'training_executed':False,
       'optimizer_steps':0,'gpu_work':False,'mock_files_removed':True})
